In [1]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()
google_api_key = os.getenv("GOOGLE_API_KEY")
print("Google API Key loaded!" if google_api_key else "Key not found")

Google API Key loaded!


In [2]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
print("API key loaded!" if api_key else "API key not found - check your .env file")

API key loaded!


In [3]:
def get_video_id(url):
    if "v=" in url:
        video_id = url.split("v=")[1]
        video_id = video_id.split("&")[0]
        return video_id
    elif "youtu.be/" in url:
        video_id = url.split("youtu.be/")[1]
        video_id = video_id.split("?")[0]
        return video_id
    else:
        return None

In [4]:
print(get_video_id("https://www.youtube.com/watch?v=aircAruvnKk&t=120"))

aircAruvnKk


In [5]:
def get_transcript(url):
    video_id = get_video_id(url)
    if not video_id:
        print("Invalid Youtube URL")
        return None
    try:
        ytt_api = YouTubeTranscriptApi()
        transcript = ytt_api.fetch(video_id)
        return transcript
    except Exception as e:
        print("Error: Could not fetch transcript. The video may not have captions or may not exist.")
        return None

In [6]:
url = "https://www.youtube.com/watch?v=aircAruvnKk"
transcript = get_transcript(url)

if transcript:
    for entry in transcript[:5]:
        print(entry)
    print(f"\nTotal segments: {len(transcript)}")

FetchedTranscriptSnippet(text='This is a 3.', start=4.22, duration=1.18)
FetchedTranscriptSnippet(text="It's sloppily written and rendered at an extremely low resolution of 28x28 pixels,", start=6.06, duration=4.653)
FetchedTranscriptSnippet(text='but your brain has no trouble recognizing it as a 3.', start=10.713, duration=3.007)
FetchedTranscriptSnippet(text='And I want you to take a moment to appreciate how', start=14.34, duration=2.219)
FetchedTranscriptSnippet(text='crazy it is that brains can do this so effortlessly.', start=16.559, duration=2.401)

Total segments: 286


In [7]:
def chunk_transcript(transcript, chunk_duration=120):
    chunks = []
    current_text = ""
    chunk_start = transcript[0].start

    for segment in transcript:
        if segment.start - chunk_start >= chunk_duration:
            chunks.append({
                "text": current_text.strip(),
                "start": chunk_start,
                "end": segment.start
            })
            current_text = segment.text + " "
            chunk_start = segment.start
        else:
            current_text += segment.text + " "

    # Don't forget the last chunk that's still being built
    if current_text:
        chunks.append({
            "text": current_text.strip(),
            "start": chunk_start,
            "end": transcript[-1].start
        })

    return chunks

In [8]:
chunks = chunk_transcript(transcript)
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} | {chunk['start']:.0f}s - {chunk['end']:.0f}s")
    print(chunk['text'][:100] + "...")
    print()

Chunk 1 | 4s - 125s
This is a 3. It's sloppily written and rendered at an extremely low resolution of 28x28 pixels, but ...

Chunk 2 | 125s - 246s
There are many many variants of neural networks, and in recent years there's been sort of a boom in ...

Chunk 3 | 246s - 368s
which for the time being should just be a giant question mark for how on earth this process of recog...

Chunk 4 | 368s - 488s
Now in a perfect world, we might hope that each neuron in the second to last layer corresponds with ...

Chunk 5 | 488s - 610s
Parsing speech, for example, involves taking raw audio and picking out distinct sounds, which combin...

Chunk 6 | 610s - 733s
are bright but the surrounding pixels are darker. When you compute a weighted sum like this, you mig...

Chunk 7 | 733s - 854s
The connections between the other layers also have a bunch of weights and biases associated with the...

Chunk 8 | 854s - 975s
By the way, so much of machine learning just comes down to having a good grasp of linear al

In [9]:
result = get_transcript("https://www.google.com")
print(result)

Invalid Youtube URL
None


In [10]:
result = get_transcript("hello this is not a url")
print(result)

Invalid Youtube URL
None


In [11]:
result = get_transcript("https://www.youtube.com/watch?v=fakevideoid123")
print(result)

Error: Could not fetch transcript. The video may not have captions or may not exist.
None


In [17]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv(override=True)
groq_api_key = os.getenv("GROQ_API_KEY")
print("Groq API Key loaded!" if groq_api_key else "Key not found")

Groq API Key loaded!


In [19]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    api_key=groq_api_key
)

response = llm.invoke("Say hello in one word")
print(response.content)

Hello


In [22]:
prompt = f"""Based on the following transcript from a lecture video, generate 3 multiple choice questions. Each question should have:
- 4 options (A, B, C, D)
- The correct answer
- An explanation of why that answer is correct

Transcript:
{chunks[0]['text']}

Return the output in the following JSON format:
{{
    "questions": [
        {{
            "question": "the question text",
            "options": {{"A": "...", "B": "...", "C": "...", "D": "..."}},
            "correct_answer": "B",
            "explanation": "why this is correct"
        }}
    ]
}}

Return ONLY the JSON, no other text.
"""

response = llm.invoke(prompt)
print(response.content)

{
    "questions": [
        {
            "question": "What is the resolution of the image of the digit 3 shown in the lecture video?",
            "options": {"A": "56x56 pixels", "B": "28x28 pixels", "C": "10x10 pixels", "D": "100x100 pixels"},
            "correct_answer": "B",
            "explanation": "The correct answer is B because the transcript states that the image of the digit 3 is 'rendered at an extremely low resolution of 28x28 pixels'."
        },
        {
            "question": "What task does the lecturer want to accomplish with a program in the video?",
            "options": {"A": "To generate handwritten digits", "B": "To recognize spoken words", "C": "To take in a grid of 28x28 pixels and output a single number between 0 and 10", "D": "To create a neural network from scratch"},
            "correct_answer": "C",
            "explanation": "The correct answer is C because the transcript states that the lecturer wants to 'write for me a program that takes in a gr

In [23]:
import json

quiz_data = json.loads(response.content)
print(type(quiz_data))
print(quiz_data["questions"][0]["question"])

<class 'dict'>
What is the resolution of the image of the digit 3 shown in the lecture video?


In [24]:
for i, q in enumerate(quiz_data["questions"]):
    print(f"Question {i+1}: {q['question']}")
    for option, text in q["options"].items():
        print(f"  {option}) {text}")
    print(f"Correct Answer: {q['correct_answer']}")
    print(f"Explanation: {q['explanation']}")
    print()

Question 1: What is the resolution of the image of the digit 3 shown in the lecture video?
  A) 56x56 pixels
  B) 28x28 pixels
  C) 10x10 pixels
  D) 100x100 pixels
Correct Answer: B
Explanation: The correct answer is B because the transcript states that the image of the digit 3 is 'rendered at an extremely low resolution of 28x28 pixels'.

Question 2: What task does the lecturer want to accomplish with a program in the video?
  A) To generate handwritten digits
  B) To recognize spoken words
  C) To take in a grid of 28x28 pixels and output a single number between 0 and 10
  D) To create a neural network from scratch
Correct Answer: C
Explanation: The correct answer is C because the transcript states that the lecturer wants to 'write for me a program that takes in a grid of 28x28 pixels like this and outputs a single number between 0 and 10, telling you what it thinks the digit is'.

Question 3: What is the main topic that the lecturer wants to cover in the two videos?
  A) The histor

In [25]:
def take_quiz(quiz_data):
    score = 0
    for i, q in enumerate(quiz_data["questions"]):
        print(f"\nQuestion {i+1}: {q['question']}")
        for option, text in q["options"].items():
            print(f"  {option}) {text}")
        user_answer = input("\nYour answer (A/B/C/D): ").upper()
        if user_answer == q['correct_answer']:
            score = score + 1
            print("Correct!")
        else:
            print("Incorrect!")
        
        # This runs for both right and wrong answers
        print(f"Correct Answer: {q['correct_answer']}")
        print(f"Explanation: {q['explanation']}")
    
    # After the loop - print final score
    print(f"\nFinal Score: {score}/{len(quiz_data['questions'])}")
        
        # Now get the user's answer
        # Compare it with the correct answer
        # Print if right or wrong
        # Show explanation

In [26]:
take_quiz(quiz_data)


Question 1: What is the resolution of the image of the digit 3 shown in the lecture video?
  A) 56x56 pixels
  B) 28x28 pixels
  C) 10x10 pixels
  D) 100x100 pixels



Your answer (A/B/C/D):  A


Incorrect!
Correct Answer: B
Explanation: The correct answer is B because the transcript states that the image of the digit 3 is 'rendered at an extremely low resolution of 28x28 pixels'.

Question 2: What task does the lecturer want to accomplish with a program in the video?
  A) To generate handwritten digits
  B) To recognize spoken words
  C) To take in a grid of 28x28 pixels and output a single number between 0 and 10
  D) To create a neural network from scratch



Your answer (A/B/C/D):  d


Incorrect!
Correct Answer: C
Explanation: The correct answer is C because the transcript states that the lecturer wants to 'write for me a program that takes in a grid of 28x28 pixels like this and outputs a single number between 0 and 10, telling you what it thinks the digit is'.

Question 3: What is the main topic that the lecturer wants to cover in the two videos?
  A) The history of machine learning
  B) The application of neural networks in computer vision
  C) The structure and learning of neural networks
  D) The mathematics behind deep learning



Your answer (A/B/C/D):  addd


Incorrect!
Correct Answer: C
Explanation: The correct answer is C because the transcript states that the lecturer wants to 'show you what a neural network actually is, assuming no background, and to help visualize what it's doing' and that 'this video is just going to be devoted to the structure component of that, and the following one is going to tackle learning'.

Final Score: 0/3


In [35]:
def generate_quiz(url, num_questions=3):
    # Step 1: Get transcript
    transcript = get_transcript(url)
    if not transcript:
        return None
    
    # Step 2: Chunk it
    chunks = chunk_transcript(transcript)
    all_text = " ".join([chunk["text"] for chunk in chunks])
    
    # Step 3: Build the prompt
    prompt = f"""Based on the following transcript from a lecture video, generate {num_questions} multiple choice questions. Each question should have:
- 4 options (A, B, C, D)
- The correct answer
- An explanation of why that answer is correct

Transcript:
{all_text}

Return the output in the following JSON format:
{{
    "questions": [
        {{
            "question": "the question text",
            "options": {{"A": "...", "B": "...", "C": "...", "D": "..."}},
            "correct_answer": "B",
            "explanation": "why this is correct"
        }}
    ]
}}

Return ONLY the JSON, no other text.
"""
    
    # Step 4: Send to LLM
    response = llm.invoke(prompt)
    
    # Step 5: Parse JSON and return
    content = response.content.strip()
    if content.startswith("```json"):
        content = content[7:]
    if content.startswith("```"):
        content = content[3:]
    if content.endswith("```"):
        content = content[:-3]
    content = content.strip()
    
    quiz_data = json.loads(content)
    return quiz_data
    

In [36]:
quiz = generate_quiz("https://www.youtube.com/watch?v=aircAruvnKk")
take_quiz(quiz)


Question 1: What is the primary function of the sigmoid function in a neural network?
  A) To increase the complexity of the network
  B) To squish the real number line into the range between 0 and 1
  C) To reduce the number of layers in the network
  D) To decrease the number of neurons in the network



Your answer (A/B/C/D):  b


Correct!
Correct Answer: B
Explanation: The sigmoid function is used to squish the real number line into the range between 0 and 1, which is necessary for the network to produce output values between 0 and 1.

Question 2: How many neurons are in the input layer of the neural network described in the video?
  A) 10
  B) 16
  C) 784
  D) 13,000



Your answer (A/B/C/D):  a


Incorrect!
Correct Answer: C
Explanation: The input layer has 784 neurons, each corresponding to one of the 28x28 pixels of the input image.

Question 3: What is the purpose of the bias in a neural network?
  A) To increase the weight of a connection
  B) To decrease the weight of a connection
  C) To determine the threshold for a neuron to be active
  D) To reduce the number of layers in the network



Your answer (A/B/C/D):  c


Correct!
Correct Answer: C
Explanation: The bias determines the threshold for a neuron to be active, i.e., how high the weighted sum needs to be before the neuron starts getting meaningfully active.

Final Score: 2/3


In [29]:
all_text = " ".join([chunk["text"] for chunk in chunks])
print(f"Total words: {len(all_text.split())}")

Total words: 3357


In [34]:
print(response.content)

{
    "questions": [
        {
            "question": "What is the resolution of the image of the digit 3 shown in the lecture video?",
            "options": {"A": "56x56 pixels", "B": "28x28 pixels", "C": "10x10 pixels", "D": "100x100 pixels"},
            "correct_answer": "B",
            "explanation": "The correct answer is B because the transcript states that the image of the digit 3 is 'rendered at an extremely low resolution of 28x28 pixels'."
        },
        {
            "question": "What task does the lecturer want to accomplish with a program in the video?",
            "options": {"A": "To generate handwritten digits", "B": "To recognize spoken words", "C": "To take in a grid of 28x28 pixels and output a single number between 0 and 10", "D": "To create a neural network from scratch"},
            "correct_answer": "C",
            "explanation": "The correct answer is C because the transcript states that the lecturer wants to 'write for me a program that takes in a gr